# HDR Credibility Check

This notebook demonstrates how to use coppuccino's HDR (Highest Density Region)
credibility to validate Bayesian inference results.

The idea: if your posterior is well-calibrated, the true (injected) parameters
should have HDR values uniformly distributed on [0, 1] across many events.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from coppuccino import compute_injection_hdr, normalizing_flows_fit, sample

## 1. Simulate a "well-calibrated" posterior

We generate synthetic posterior samples centered on the true parameters,
mimicking what a correct Bayesian analysis would produce.

In [ ]:
np.random.seed(0)

# True (injected) parameters
true_params = np.array([2.0, -1.0, 0.5])

# Simulated posterior: correlated Gaussian centered near the truth
cov = [
    [1.0, 0.6, -0.2],
    [0.6, 0.8,  0.3],
    [-0.2, 0.3, 0.5],
]
posterior_samples = np.random.multivariate_normal(true_params, cov, 3000)

print(f"Posterior shape: {posterior_samples.shape}")
print(f"True params: {true_params}")

## 2. Compute HDR credibility for the true parameters

In [ ]:
hdr = compute_injection_hdr(posterior_samples, true_params, num_samples=50_000, max_epochs=200)
print(f"HDR credibility: {hdr[0]:.3f}")
# The injection sits at the posterior peak, so its HDR is small (near 0).
# A point in the tails would give HDR near 1. Calibration (uniformity on
# [0, 1]) is something you assess across MANY events, as in section 3 below.

## 3. HDR calibration over many events

To check calibration properly, we repeat the analysis across many
simulated events. If the inference is well-calibrated, the HDR values
should follow a uniform distribution.

In [ ]:
n_events = 20
hdrs = []

for i in range(n_events):
    # Each event: new true params drawn from the prior, new posterior around them
    rng = np.random.RandomState(i)
    event_truth = rng.randn(3) * 2  # draw from a broad prior
    event_posterior = rng.multivariate_normal(event_truth, cov, 2000)

    h = compute_injection_hdr(event_posterior, event_truth, num_samples=50_000,
                              max_epochs=150, rng_seed=i)
    hdrs.append(h[0])
    print(f"Event {i+1:2d}: HDR = {h[0]:.3f}")

hdrs = np.array(hdrs)

## 4. Plot the P-P plot

A P-P plot of HDR values should lie close to the diagonal for well-calibrated inference.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# P-P plot
sorted_hdrs = np.sort(hdrs)
expected = np.linspace(0, 1, len(sorted_hdrs))
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5, label="Ideal")
axes[0].step(sorted_hdrs, expected, where="post", label=f"Observed (n={n_events})")
axes[0].set_xlabel("HDR credibility")
axes[0].set_ylabel("Cumulative fraction")
axes[0].set_title("P-P Plot")
axes[0].legend()

# Histogram
axes[1].hist(hdrs, bins=10, range=(0, 1), density=True, alpha=0.7)
axes[1].axhline(1.0, color="k", linestyle="--", alpha=0.5, label="Uniform")
axes[1].set_xlabel("HDR credibility")
axes[1].set_ylabel("Density")
axes[1].set_title("HDR Distribution")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Evaluating multiple injection points at once

You can also pass multiple injection points to a single posterior.

In [ ]:
# Three injection points: one at the mean, one offset, one far away
injections = np.array([
    true_params,                    # at the truth
    true_params + np.array([1, 1, 1]),  # offset
    true_params + np.array([5, 5, 5]),  # far from truth (should be HDR ~ 1)
])

multi_hdrs = compute_injection_hdr(posterior_samples, injections, num_samples=50_000, max_epochs=200)
for i, h in enumerate(multi_hdrs):
    print(f"Injection {i+1}: HDR = {h:.3f}")